# WPFormer — CVPR 2025 — baseline reproduction on CrackSeg9k

**Paper:** *Wavelet and Prototype Augmented Query-based Transformer for Pixel-level Surface Defect Detection*
Yan, Jiang, Lu, Cao, Chen, Xu — CVPR 2025, pp. 23860–23869
**Official code:** https://github.com/fengyan-cv/WPFormer (pinned to commit `83a33bb`)

---

### What this notebook does

Runs the authors' **original, unmodified code** on Colab and reports the paper's five metrics
on the CrackSeg9k test set. Nothing in the model, the metrics, or the evaluation protocol is
changed — no improvements from our project are applied here. This is the reference point
that everything later has to beat.

`Runtime → Run all`. No edits required.

### Read this before you run

The repo was written for Python 3.7 / PyTorch 1.11 on the author's **Windows** machine, and it
carries hard-coded absolute paths such as `D:\yanfeng\Paper Code\CVPR2025\WPFormer\model\pvt_v2_b2.pth`.
Rather than editing those lines out, this notebook makes them resolve on Linux: on POSIX a
backslash is an ordinary filename character, so `D:\yanfeng\...\pvt_v2_b2.pth` is simply a
*relative filename that happens to contain backslashes*. We create files and folders with
exactly those names, read straight out of the repo source with `ast.literal_eval`, so the
authors' `.py` files run byte-for-byte as shipped.

Three environment-level shims are also installed (**no repo file is touched**):

| Shim | Why |
|---|---|
| `timm.models.layers` / `timm.models.registry` aliases | those module paths moved in modern `timm` |
| stub `mmcv.cnn.get_model_complexity_info` | imported at the top of `defect_test.py`, never called |
| `torch.load(..., weights_only=False)` default | PyTorch ≥ 2.6 flipped this default |

The last one un-pickles the checkpoints, which executes code from the file. The checkpoints
come from the Google Drive links in the official repo's README — that is the normal risk of any
research checkpoint, just be aware of it.

### Target numbers — paper Table 1, CrackSeg9k row

| MAE ↓ | wF<sub>β</sub> ↑ | S<sub>α</sub> ↑ | mF<sub>β</sub> ↑ | mE<sub>ξ</sub> ↑ |
|---|---|---|---|---|
| .0135 | .7672 | .8493 | .7679 | .9481 |

Config for this row: PVTv2-B2 backbone, channel=64, 16 queries, 384×384, bs=4, lr=8e-5, 60 epochs.
CrackSeg9k = 7243 train / **395 test** images.

## 1 · Environment check

Needs a GPU runtime: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import sys, torch, platform
print("python      :", sys.version.split()[0], "|", platform.platform())
print("torch       :", torch.__version__, "| cuda", torch.version.cuda)
print("gpu available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         :", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")

## 2 · Clone the official repository (pinned commit)

Pinned so every run of this notebook — and every member of the team — gets identical source.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/fengyan-cv/WPFormer.git"
COMMIT   = "83a33bbf5ed96dff069e9d58f5f3e0c464bae446"   # main @ 2026-04-20
REPO_DIR = "/content/WPFormer"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", COMMIT], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("commit:", subprocess.run(["git","-C",REPO_DIR,"log","-1","--pretty=%H  %ad  %s"],
                                capture_output=True, text=True).stdout.strip())
print("cwd   :", os.getcwd())
print("files :", sorted(os.listdir(".")))

## 3 · Dependencies

Colab already ships torch, torchvision, opencv, scikit-image, scipy and matplotlib.
Only `gdown` (Google Drive downloads) and `timm` (PVTv2 building blocks) are added.

In [ ]:
!pip install -q gdown timm
import gdown, timm
print("gdown", gdown.__version__, "| timm", timm.__version__)

## 4 · Compatibility shims

Applied to this kernel **and** written to `/content/_shims` so that a subprocess
`python defect_test.py` (section 11) picks them up too, via `PYTHONPATH` + `sitecustomize`.
No file inside the repo is modified.

In [ ]:
import sys, os, types, textwrap, importlib, torch

SHIM_DIR = "/content/_shims"
os.makedirs(SHIM_DIR, exist_ok=True)

# ---- (a) timm legacy module paths --------------------------------------------------
def _install_timm_aliases():
    import timm, timm.models
    try:
        from timm.models.layers import DropPath, to_2tuple, trunc_normal_   # noqa
    except Exception:
        import timm.layers as _tl
        m = types.ModuleType("timm.models.layers")
        for n in dir(_tl):
            if not n.startswith("__"):
                setattr(m, n, getattr(_tl, n))
        sys.modules["timm.models.layers"] = m; timm.models.layers = m
    try:
        from timm.models.registry import register_model                      # noqa
    except Exception:
        try:    from timm.models._registry import register_model as _rm
        except Exception: from timm.models import register_model as _rm
        m = types.ModuleType("timm.models.registry"); m.register_model = _rm
        sys.modules["timm.models.registry"] = m; timm.models.registry = m

_install_timm_aliases()
from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from timm.models.registry import register_model
try:
    from timm.models.vision_transformer import _cfg                          # noqa
except Exception:                       # very new timm dropped the private helper
    import timm.models.vision_transformer as _vt
    _vt._cfg = lambda url="", **kw: {"url": url, "num_classes": 1000,
                                     "input_size": (3, 224, 224), **kw}
print("[ok] timm legacy import paths available")

# ---- (b) mmcv stub  (defect_test.py imports it at module level, never calls it) -----
try:
    from mmcv.cnn import get_model_complexity_info      # noqa
    print("[ok] real mmcv present")
except Exception:
    os.makedirs(f"{SHIM_DIR}/mmcv/cnn", exist_ok=True)
    open(f"{SHIM_DIR}/mmcv/__init__.py", "w").write("__version__ = '0.0.0-stub'\n")
    open(f"{SHIM_DIR}/mmcv/cnn/__init__.py", "w").write(textwrap.dedent('''
        # Minimal stand-in: WPFormer only imports get_model_complexity_info, never calls it.
        def get_model_complexity_info(model, input_shape, *a, **kw):
            n = sum(p.numel() for p in model.parameters())
            return "n/a (mmcv stub)", f"{n/1e6:.2f} M"
    ''').lstrip())
    if SHIM_DIR not in sys.path: sys.path.insert(0, SHIM_DIR)
    from mmcv.cnn import get_model_complexity_info      # noqa
    print("[ok] mmcv stub installed at", SHIM_DIR)

# ---- (c) torch.load default (PyTorch >= 2.6 flipped weights_only to True) -----------
if not getattr(torch.load, "_wpformer_patched", False):
    _orig_load = torch.load
    def _load(*a, **kw):
        kw.setdefault("weights_only", False)
        return _orig_load(*a, **kw)
    _load._wpformer_patched = True
    torch.load = _load
print("[ok] torch.load defaults to weights_only=False")

# ---- (d) same shims for subprocess runs --------------------------------------------
open(f"{SHIM_DIR}/sitecustomize.py", "w").write(textwrap.dedent('''
    import sys, types, torch
    try:
        import timm, timm.models
        try: from timm.models.layers import DropPath
        except Exception:
            import timm.layers as _tl
            m = types.ModuleType("timm.models.layers")
            for n in dir(_tl):
                if not n.startswith("__"): setattr(m, n, getattr(_tl, n))
            sys.modules["timm.models.layers"] = m; timm.models.layers = m
        try: from timm.models.registry import register_model
        except Exception:
            try: from timm.models._registry import register_model as _rm
            except Exception: from timm.models import register_model as _rm
            m = types.ModuleType("timm.models.registry"); m.register_model = _rm
            sys.modules["timm.models.registry"] = m; timm.models.registry = m
    except Exception as e:
        print("sitecustomize timm shim skipped:", e)
    _orig = torch.load
    def _load(*a, **kw):
        kw.setdefault("weights_only", False); return _orig(*a, **kw)
    torch.load = _load
''').lstrip())
os.environ["PYTHONPATH"] = SHIM_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")
print("[ok] sitecustomize written; PYTHONPATH =", os.environ["PYTHONPATH"])

## 5 · Download the checkpoints and the dataset

All three Google Drive IDs are taken verbatim from the official README table
(CrackSeg9k row). If a download stalls on Drive's daily quota, open the link in a
browser, add the file to your own Drive, and re-run — or mount Drive and point
`DL_DIR` at it.

In [ ]:
import os, gdown, zipfile, tarfile, shutil, glob

DL_DIR   = "/content/downloads"
CKPT_DIR = "/content/checkpoints"
os.makedirs(DL_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)

# Drive IDs — official README, CrackSeg9k row
ID_BACKBONE   = "1o3PDfaIKlx1EB21lbt_h37nRzwzJYoIX"   # PVTv2-B2 ImageNet weights
ID_CHECKPOINT = "17Yq3nr3CoxGL0P6hXdWnWmDo3yiCYzVU"   # WPFormer trained on CrackSeg9k
ID_DATASET    = "1pOQBOjs_r9g6by0QQWU6hFT-dGeHlQqZ"   # CrackSeg9k images + masks

def fetch(file_id, out_path=None, out_dir=None):
    # Download one Drive file; skip if already present. Returns the local path.
    if out_path and os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print("cached:", out_path); return out_path
    if out_dir:
        got = gdown.download(id=file_id, output=out_dir + "/", quiet=False, resume=True)
    else:
        got = gdown.download(id=file_id, output=out_path, quiet=False, resume=True)
    if not got:
        raise RuntimeError(
            f"Drive download failed for id={file_id}. Most likely the daily quota was hit.\n"
            f"Open https://drive.google.com/file/d/{file_id}/view , copy it to your own Drive, "
            f"then mount Drive and place the file under {DL_DIR}.")
    return got

pvt_path  = fetch(ID_BACKBONE,   out_path=f"{CKPT_DIR}/pvt_v2_b2.pth")
ckpt_path = fetch(ID_CHECKPOINT, out_path=f"{CKPT_DIR}/CrackSeg9k.pth")

ARCH_EXT = (".zip", ".tar", ".gz", ".tgz", ".rar", ".7z")
existing = [p for p in glob.glob(f"{DL_DIR}/*")
            if os.path.isfile(p) and p.lower().endswith(ARCH_EXT)
            and os.path.getsize(p) > 1e6]
data_archive = existing[0] if existing else fetch(ID_DATASET, out_dir=DL_DIR)

for p in (pvt_path, ckpt_path, data_archive):
    print(f"{os.path.getsize(p)/1e6:9.1f} MB  {p}")

In [ ]:
# ---- extract the dataset archive ----------------------------------------------------
EXTRACT_DIR = "/content/datasets_raw"

if not os.path.isdir(EXTRACT_DIR) or not os.listdir(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    if zipfile.is_zipfile(data_archive):
        with zipfile.ZipFile(data_archive) as z:
            z.extractall(EXTRACT_DIR)
    elif tarfile.is_tarfile(data_archive):
        with tarfile.open(data_archive) as t:
            t.extractall(EXTRACT_DIR)
    else:
        raise RuntimeError(f"Unrecognised archive: {data_archive}")
    print("extracted ->", EXTRACT_DIR)
else:
    print("already extracted ->", EXTRACT_DIR)

for root, dirs, files in os.walk(EXTRACT_DIR):
    depth = root[len(EXTRACT_DIR):].count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(root) + f"/   [{len(files)} files]")
    if depth >= 3:
        dirs[:] = []

## 6 · Locate and normalise the test split

The archive's internal folder names are not documented, so instead of assuming a layout we
search for the `test` split and its image / mask folders under the usual spellings, then expose
a canonical `/content/data/CrackSeg9k/test/{images,gt}` via symlinks.

Two guards run here, and both matter for a *faithful* reproduction:

* the test set must contain **395** images (the count stated in the paper);
* image and mask filenames must line up after sorting — `defect_test.py` pairs them by
  `sorted(os.listdir(...))`, so a stem mismatch silently produces meaningless metrics.

In [ ]:
import os, glob, re
from pathlib import Path

IMG_NAMES = ["images", "image", "img", "imgs", "Images", "Image", "JPEGImages"]
GT_NAMES  = ["gt", "GT", "gts", "masks", "mask", "labels", "label", "annotations", "Masks"]
EXTS      = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".JPG", ".PNG")

def n_images(d):
    return len([f for f in os.listdir(d) if f.endswith(EXTS)]) if os.path.isdir(d) else 0

def find_split(root, split_keyword):
    # Return (image_dir, gt_dir) for a split, or None if not found.
    cands = []
    for dirpath, dirnames, _ in os.walk(root):
        if split_keyword.lower() not in os.path.basename(dirpath).lower():
            continue
        img_d = next((os.path.join(dirpath, n) for n in IMG_NAMES
                      if n_images(os.path.join(dirpath, n)) > 0), None)
        gt_d  = next((os.path.join(dirpath, n) for n in GT_NAMES
                      if n_images(os.path.join(dirpath, n)) > 0), None)
        if img_d and gt_d:
            cands.append((n_images(img_d), img_d, gt_d))
    if not cands:
        return None
    cands.sort(reverse=True)
    return cands[0][1], cands[0][2]

found = find_split(EXTRACT_DIR, "test") or find_split(EXTRACT_DIR, "val")
if found is None:
    raise RuntimeError(
        "Could not locate a test split. Inspect the tree printed above and set "
        "TEST_IMAGES / TEST_GT by hand in the next line.")
TEST_IMAGES, TEST_GT = found

DATA_ROOT = "/content/data/CrackSeg9k/test"
os.makedirs(DATA_ROOT, exist_ok=True)
for link, target in (("images", TEST_IMAGES), ("gt", TEST_GT)):
    dst = os.path.join(DATA_ROOT, link)
    if os.path.islink(dst):
        os.unlink(dst)
    if not os.path.exists(dst):
        os.symlink(os.path.abspath(target), dst)

test_image_root = DATA_ROOT + "/images/"     # trailing slash: the repo concatenates strings
test_gt_root    = DATA_ROOT + "/gt/"

imgs = sorted(f for f in os.listdir(test_image_root) if f.endswith(EXTS))
gts  = sorted(f for f in os.listdir(test_gt_root)    if f.endswith(EXTS))

print("images :", test_image_root, "->", len(imgs), "files")
print("masks  :", test_gt_root,    "->", len(gts),  "files")

assert len(imgs) == len(gts), f"count mismatch: {len(imgs)} images vs {len(gts)} masks"
if len(imgs) != 395:
    print(f"\n[warn] paper states 395 CrackSeg9k test images, found {len(imgs)} — "
          f"metrics will not be directly comparable to Table 1.")
else:
    print("\n[ok] 395 test images, matches the paper.")

bad = [(a, b) for a, b in zip(imgs, gts) if Path(a).stem != Path(b).stem]
if bad:
    print(f"\n[warn] {len(bad)} image/mask pairs disagree after sorting, e.g. {bad[:3]}")
    print("       defect_test.py pairs by sorted order — metrics would be meaningless.")
else:
    print("[ok] every image is paired with the matching mask.")

## 7 · Satisfy the hard-coded backbone path

`model/WPFormer.py` line 267 does:

```python
path = 'D:\yanfeng\Paper Code\CVPR2025\WPFormer\model\pvt_v2_b2.pth'
save_model = torch.load(path)
```

On Linux that string is one relative filename containing backslashes, so we read the literal
straight out of the source with `ast.literal_eval` (which resolves the escapes exactly the way
Python's own parser does) and drop the downloaded weights there. **The file is not edited.**

In [ ]:
import ast, re, shutil, warnings

with open("model/WPFormer.py", encoding="utf-8", errors="replace") as f:
    wp_src = f.read()

m = re.search(r"path\s*=\s*('[^']*pvt_v2_b2\.pth')", wp_src)
assert m, "could not find the pvt_v2_b2 path literal in model/WPFormer.py"

with warnings.catch_warnings():
    warnings.simplefilter("ignore")           # invalid \escapes are intentional here
    backbone_literal = ast.literal_eval(m.group(1))

print("literal in source :", repr(backbone_literal))

parent = os.path.dirname(backbone_literal)    # '' — no '/' in the string
if parent:
    os.makedirs(parent, exist_ok=True)
if not os.path.exists(backbone_literal):
    shutil.copyfile(pvt_path, backbone_literal)

print("created           :", repr(os.path.abspath(backbone_literal)))
print("size              :", f"{os.path.getsize(backbone_literal)/1e6:.1f} MB")
print("torch.load works  :", isinstance(torch.load(backbone_literal, map_location='cpu'), dict))

## 8 · Build WPFormer and load the CrackSeg9k checkpoint

Exactly the constructor call from `defect_test.py`: `WPFormer(method="pvt_v2_b2", channel=64)`,
which defaults to `num_queries=16` — the paper's CrackSeg9k configuration.

The repo loads with `strict=False`, which will happily accept a checkpoint that barely matches.
We print the missing / unexpected key counts so a bad load can't masquerade as a bad model.

In [ ]:
import torch, time
from model.WPFormer import WPFormer

net = WPFormer(method="pvt_v2_b2", channel=64).cuda()

state = torch.load(ckpt_path, map_location="cuda")
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

incompat = net.load_state_dict(state, strict=False)
missing, unexpected = list(incompat.missing_keys), list(incompat.unexpected_keys)

n_model = len(net.state_dict())
print(f"checkpoint tensors : {len(state)}")
print(f"model tensors      : {n_model}")
print(f"missing keys       : {len(missing)}    (in model, absent from checkpoint)")
print(f"unexpected keys    : {len(unexpected)} (in checkpoint, absent from model)")
if missing[:5]:    print("  e.g. missing   :", missing[:5])
if unexpected[:5]: print("  e.g. unexpected:", unexpected[:5])

if len(missing) > 0.02 * n_model:
    print("\n[warn] a substantial part of the model is NOT covered by the checkpoint — "
          "metrics below will be far from the paper.")
else:
    print("\n[ok] checkpoint covers the model.")

n_par = sum(p.numel() for p in net.parameters())
print(f"\nparameters         : {n_par/1e6:.2f} M")

net.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, 384, 384).cuda()
    out = net(dummy)
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(20): net(dummy)
    torch.cuda.synchronize()
print(f"forward outputs    : {len(out)} maps, final shape {tuple(out[-1].shape)}")
print(f"inference          : {(time.time()-t0)/20*1000:.1f} ms/image @384x384 "
      f"({20/(time.time()-t0):.1f} FPS)")

## 9 · Evaluate — the paper's protocol, verbatim

The loop below is `eval_psnr` from `defect_test.py` with its logic untouched: same resize to
384×384, same ImageNet normalisation, same `pred[-1]`, same sigmoid → min-max normalise →
`Image.fromarray(pred*255).convert("L")` → bilinear resize back to the ground-truth resolution.
Metrics come from the repo's own `sod_metrics.py` (the PySODMetrics implementation).

Only two things are added, neither of which touches a number: a progress bar, and saving the
prediction maps to `/content/preds` for the qualitative figure and for the repo's `eval.py`.

~395 images, a couple of minutes on a T4.

In [ ]:
import numpy as np, cv2, torch, os
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm
from sod_metrics import MAE, Emeasure, Fmeasure, Smeasure, WeightedFmeasure

PRED_DIR = "/content/preds/CrackSeg9k"
os.makedirs(PRED_DIR, exist_ok=True)

def eval_psnr(test_image_root, test_gt_root, train_size, model, save_dir=None):
    FM, WFM, SM, EM, M = Fmeasure(), WeightedFmeasure(), Smeasure(), Emeasure(), MAE()

    img_transform = transforms.Compose([
        transforms.Resize((train_size, train_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    images = [test_image_root + f for f in os.listdir(test_image_root)]
    gts    = [test_gt_root    + p for p in os.listdir(test_gt_root)]
    images = sorted(images)
    gts    = sorted(gts)

    model.eval()

    for i_test in tqdm(range(len(images)), desc="evaluating"):
        ori_image = Image.open(images[i_test]).convert("RGB")
        image = img_transform(ori_image).unsqueeze(0).cuda()
        gt = cv2.imread(gts[i_test], cv2.IMREAD_GRAYSCALE)
        H, W = gt.shape

        with torch.no_grad():
            pred = model(image)
            res  = pred[-1]
            res  = torch.sigmoid(res).data.cpu().numpy().squeeze()

        pred = (res - res.min()) / (res.max() - res.min() + 1e-8)
        pred = Image.fromarray(pred * 255).convert("L")
        pred = pred.resize((W, H), resample=Image.BILINEAR)

        if save_dir:
            pred.save(os.path.join(save_dir,
                      os.path.splitext(os.path.basename(images[i_test]))[0] + ".png"))

        pred = np.array(pred)
        FM.step(pred=pred, gt=gt); WFM.step(pred=pred, gt=gt); SM.step(pred=pred, gt=gt)
        EM.step(pred=pred, gt=gt); M.step(pred=pred, gt=gt)

    fm  = FM.get_results()["fm"]
    wfm = WFM.get_results()["wfm"]
    sm  = SM.get_results()["sm"]
    em  = EM.get_results()["em"]
    mae = M.get_results()["mae"]

    return {
        "MAE":       float(mae),
        "wFmeasure": float(wfm),
        "Smeasure":  float(sm),
        "meanFm":    float(fm["curve"].mean()),
        "meanEm":    float(em["curve"].mean()),
    }, FM.get_results(), EM.get_results()

results, fm_all, em_all = eval_psnr(test_image_root, test_gt_root, 384, net, save_dir=PRED_DIR)
print("\n", {k: f"{v:.4f}" for k, v in results.items()})

## 10 · Reproduction check against Table 1

`Δ` is *our number minus the paper's*. For MAE lower is better, so a negative Δ there is good;
for the other four a positive Δ is good. Anything within roughly ±0.005 is normal
reproduction noise from library versions and JPEG decoding.

**Whatever comes out of this table is our baseline** — the improvements in the project are
measured against this, not against the printed paper values.

In [ ]:
PAPER = {"MAE": .0135, "wFmeasure": .7672, "Smeasure": .8493, "meanFm": .7679, "meanEm": .9481}
LOWER_IS_BETTER = {"MAE"}

hdr = f"{'metric':<12}{'ours':>10}{'paper':>10}{'delta':>10}   verdict"
print(hdr); print("-" * len(hdr))
for k in ["MAE", "wFmeasure", "Smeasure", "meanFm", "meanEm"]:
    ours, paper = results[k], PAPER[k]
    d = ours - paper
    good = (d < 0) if k in LOWER_IS_BETTER else (d > 0)
    verdict = "matches" if abs(d) <= 0.005 else ("better" if good else "below paper")
    print(f"{k:<12}{ours:>10.4f}{paper:>10.4f}{d:>+10.4f}   {verdict}")

print("\nconfig: PVTv2-B2 | channel=64 | 16 queries | 384x384 | CrackSeg9k test split")
print(f"images evaluated: {len(os.listdir(test_image_root))}")

import json
os.makedirs("/content/results", exist_ok=True)
with open("/content/results/baseline_crackseg9k.json", "w") as f:
    json.dump({"config": "WPFormer PVTv2-B2 ch64 q16 384", "commit": COMMIT,
               "n_test": len(os.listdir(test_image_root)),
               "ours": results, "paper": PAPER}, f, indent=2)
print("\nsaved -> /content/results/baseline_crackseg9k.json")

## 11 · Qualitative results

Input, ground truth, raw probability map, and the binarised prediction at threshold 0.5.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

img_files = sorted(os.listdir(test_image_root))
gt_files  = sorted(os.listdir(test_gt_root))
N = 5
idx = np.linspace(0, len(img_files) - 1, N).astype(int)

fig, axes = plt.subplots(N, 4, figsize=(13, 3.1 * N))
for row, i in enumerate(idx):
    img  = np.array(Image.open(test_image_root + img_files[i]).convert("RGB"))
    gt   = cv2.imread(test_gt_root + gt_files[i], cv2.IMREAD_GRAYSCALE)
    pr   = np.array(Image.open(os.path.join(PRED_DIR, Path(img_files[i]).stem + ".png")))
    binr = (pr > 127).astype(np.uint8) * 255

    for col, (im, title, cmap) in enumerate([
            (img, "image", None), (gt, "ground truth", "gray"),
            (pr, "prediction (prob)", "gray"), (binr, "prediction @0.5", "gray")]):
        ax = axes[row, col]; ax.imshow(im, cmap=cmap); ax.axis("off")
        if row == 0: ax.set_title(title, fontsize=11)
    axes[row, 0].text(0.02, 0.02, img_files[i][:22], transform=axes[row, 0].transAxes,
                      fontsize=7, color="yellow", va="bottom")

plt.suptitle("WPFormer baseline — CrackSeg9k test set", fontsize=13, y=0.995)
plt.tight_layout(); plt.savefig("/content/results/qualitative_baseline.png", dpi=130,
                                bbox_inches="tight")
plt.show()
print("saved -> /content/results/qualitative_baseline.png")

## 12 · Optional — run `defect_test.py` completely unmodified

Everything above imports the repo's modules but drives them from the notebook. This cell
instead executes the authors' script as a subprocess with **zero source edits**, by
materialising on disk the exact Windows-shaped names it expects. Every literal is pulled out of
the script with `ast.literal_eval`, so the names are right by construction rather than by
transcription.

There is one wrinkle worth understanding, because it is the reason this is not a two-line cell.
`defect_test.py` builds its file list by **string concatenation**, not by `os.path.join`:

```python
test_image_root = os.path.join(file_dir, dataset_name + "\\test\\images\\")   # -> '.\datasets\/CrackSeg9k\test\images\'
images = [test_image_root + f for f in os.listdir(test_image_root)]
```

On Linux that root is two path components: a directory `.\datasets\` containing a directory
`CrackSeg9k\test\images\`. `os.listdir` therefore works — but `root + f` appends to the
*directory's own name*, producing `CrackSeg9k\test\images\a.jpg`, which is a **sibling** of that
directory, not a file inside it. So we create both: the directory, so the listing succeeds, and
one flat symlink per image under the concatenated name, so the reads succeed.

Also a property of the original code rather than of this notebook: line 107 hard-codes
`dataset_names = ["ZJU-Leaper", "ESDIs-SOD", "CrackSeg9k"]`, so the script walks all three and
raises on the first one you have not downloaded. Only CrackSeg9k is wired up here, so expect
tracebacks for the other two before it reaches the row we care about.

In [ ]:
import ast, re, os, warnings, subprocess

with open("defect_test.py", encoding="utf-8", errors="replace") as f:
    dt_src = f.read()

def literal(pattern, src):
    m = re.search(pattern, src)
    assert m, f"pattern not found: {pattern}"
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")     # the invalid \escapes are the point
        return ast.literal_eval(m.group(1))

save_dir_literal = literal(r'model_save\s*=\s*os\.path\.join\(\s*("[^"]*")', dt_src)
file_dir_literal = literal(r'file_dir\s*=\s*("[^"]*")', dt_src)
img_leaf         = literal(r'test_image_root\s*=\s*os\.path\.join\(file_dir,'
                           r'\s*dataset_name\s*\+\s*("[^"]*")', dt_src)
gt_leaf          = literal(r'test_gt_root\s*=\s*os\.path\.join\(file_dir,'
                           r'\s*dataset_name\s*\+\s*("[^"]*")', dt_src)

print("checkpoint dir :", repr(save_dir_literal))
print("dataset   dir  :", repr(file_dir_literal))
print("image leaf     :", repr(img_leaf))
print("gt    leaf     :", repr(gt_leaf))

# --- checkpoint: os.path.join(save_dir, "CrackSeg9k.pth") -> real dir + real file ----
os.makedirs(save_dir_literal, exist_ok=True)
dst_ckpt = os.path.join(save_dir_literal, "CrackSeg9k.pth")
if not os.path.lexists(dst_ckpt):
    os.symlink(os.path.abspath(ckpt_path), dst_ckpt)
print("\ncheckpoint ->", repr(dst_ckpt), "ok:", os.path.exists(dst_ckpt))

# --- data: a directory for os.listdir + flat siblings for the concatenation ---------
def materialise(leaf, real_dir):
    listing_dir = os.path.join(file_dir_literal, "CrackSeg9k" + leaf)
    os.makedirs(listing_dir, exist_ok=True)
    n = 0
    for f in sorted(os.listdir(real_dir)):
        src = os.path.abspath(os.path.join(real_dir, f))
        for dst in (os.path.join(listing_dir, f),                             # for listdir
                    os.path.join(file_dir_literal, "CrackSeg9k" + leaf + f)):  # for root + f
            if not os.path.lexists(dst):
                os.symlink(src, dst)
        n += 1
    return listing_dir, n

for leaf, real in ((img_leaf, test_image_root), (gt_leaf, test_gt_root)):
    d, n = materialise(leaf, real.rstrip("/"))
    print(f"linked {n:4d} files -> {d!r}")

print("\n--- running: python defect_test.py  (source untouched) ---\n")
proc = subprocess.run([sys.executable, "defect_test.py"],
                      capture_output=True, text=True, env={**os.environ})
print(proc.stdout[-4000:] or "(no stdout)")
if proc.returncode != 0:
    print("--- stderr (tail) ---")
    print(proc.stderr[-2500:])
    print("\nA FileNotFoundError on ZJU-Leaper / ESDIs-SOD is expected: only CrackSeg9k "
          "was downloaded. The CrackSeg9k dict above is the one to read.")

## 13 · Extra — IoU and Dice

Not part of the original script. The paper reports only the five saliency-style metrics, but
the crack-segmentation literature that we compare against reports IoU / Dice, and our project
uses them as a primary metric, so they are computed here on the saved prediction maps at a
sweep of thresholds. Nothing above depends on this cell.

In [ ]:
import numpy as np, cv2, os, json
from pathlib import Path

def iou_dice(pred_dir, gt_root, thresholds=np.arange(0.3, 0.75, 0.05)):
    gt_files = sorted(os.listdir(gt_root))
    inter = np.zeros(len(thresholds)); union = np.zeros(len(thresholds))
    psum  = np.zeros(len(thresholds)); gsum  = np.zeros(len(thresholds))
    for gf in gt_files:
        gt = cv2.imread(gt_root + gf, cv2.IMREAD_GRAYSCALE)
        pp = os.path.join(pred_dir, Path(gf).stem + ".png")
        if not os.path.exists(pp):
            continue
        pr = cv2.imread(pp, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        g  = (gt > 127)
        for j, t in enumerate(thresholds):
            p = pr > t
            inter[j] += np.logical_and(p, g).sum()
            union[j] += np.logical_or(p, g).sum()
            psum[j]  += p.sum(); gsum[j] += g.sum()
    return thresholds, inter / np.maximum(union, 1), 2 * inter / np.maximum(psum + gsum, 1)

ths, ious, dices = iou_dice(PRED_DIR, test_gt_root)
print(f"{'thresh':>8}{'IoU':>10}{'Dice':>10}")
for t, i, d in zip(ths, ious, dices):
    print(f"{t:>8.2f}{i:>10.4f}{d:>10.4f}")
best = int(np.argmax(ious))
print(f"\nbest IoU  {ious[best]:.4f} @ threshold {ths[best]:.2f}   (Dice {dices[best]:.4f})")
print(f"IoU @0.50 {ious[np.argmin(abs(ths-0.5))]:.4f}   "
      f"Dice @0.50 {dices[np.argmin(abs(ths-0.5))]:.4f}")

with open("/content/results/baseline_crackseg9k.json") as f: blob = json.load(f)
blob["iou_dice"] = {"thresholds": ths.tolist(), "iou": ious.tolist(), "dice": dices.tolist(),
                    "best_iou": float(ious[best]), "best_threshold": float(ths[best])}
with open("/content/results/baseline_crackseg9k.json", "w") as f: json.dump(blob, f, indent=2)
print("\nupdated -> /content/results/baseline_crackseg9k.json")

## 14 · Optional — training smoke test

Not needed to reproduce the numbers above; it only proves the training half of the repo also
runs here, so the later experiments have somewhere to start. It uses the repo's own
`data_loader.get_loader` and the authors' `total_loss` from `defect_train.py`
(BCE + IoU loss, applied to **every** output map — deep supervision), and stops after a
handful of iterations.

Full training is 60 epochs, batch 4, lr 8e-5, cosine schedule — several hours on a T4.

In [ ]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    import torch.nn as nn, torch.optim as optim
    from data_loader import get_loader

    train_found = find_split(EXTRACT_DIR, "train")
    if train_found is None:
        print("no train split found in the archive — skipping")
    else:
        tr_img, tr_gt = train_found[0] + "/", train_found[1] + "/"
        print("train images:", tr_img, len(os.listdir(tr_img)))

        loader = get_loader(tr_img, tr_gt, batchsize=2, trainsize=384, is_train=True)

        def total_loss(pred, mask):                      # verbatim from defect_train.py
            pred = torch.sigmoid(pred)
            bce  = nn.BCELoss()(pred, mask)
            inter = (pred * mask).sum(dim=(2, 3))
            union = (pred + mask).sum(dim=(2, 3))
            iou   = (1 - inter / (union - inter)).mean()
            return iou + bce

        net.train()
        opt = optim.Adam(net.parameters(), lr=8e-5)
        for it, data in enumerate(loader):
            images = data["image"].type(torch.FloatTensor).cuda()
            gts    = data["label"].type(torch.FloatTensor).cuda()
            opt.zero_grad()
            preds = net(images)
            loss = sum(total_loss(p, gts) for p in preds)
            loss.backward(); opt.step()
            print(f"iter {it}  loss {loss.item():.4f}  ({len(preds)} supervised outputs)")
            if it >= 4:
                break
        print("\n[ok] training loop runs — reload the checkpoint before evaluating again.")
        net.eval()
else:
    print("skipped")

---

## Where this leaves us

`/content/results/baseline_crackseg9k.json` holds the reproduced baseline; `/content/preds`
holds the prediction maps. Download both before the runtime recycles — Colab wipes `/content`.

```python
from google.colab import files
!zip -qr /content/wpformer_baseline.zip /content/results /content/preds
files.download("/content/wpformer_baseline.zip")
```

A caveat worth writing into the report: CrackSeg9k ships one fixed 7243 / 395 train-test split
and this notebook evaluates on that shipped test set — there is no separate validation split
yet. Carving a validation set out of the training images, and freezing it, is Division A's
first job, because tuning loss weights on the test set would invalidate every comparison
that follows.